# Validação do Painel Unificado — Inspeção visual

Lê os CSVs gerados por `scripts/validar_painel_unificado.py` e produz heatmaps, séries sobrepostas, scatter painel × gabarito e a tabela das piores divergências.

**Pré-requisito**: rodar `python scripts/validar_painel_unificado.py` antes (gera `outputs/diagnosticos/validacao_painel.csv` e `validacao_painel_resumo.csv`).

Esquema dos flags:

- **OK** — diff dentro da tolerância do bloco.
- **ESPERADA_TOCANTINS** — divergência pré-1989 com fonte que reflete o GO histórico (com TO).
- **ESPERADA_CENSO** — população 2007/2010 (IBGE não publica estimativa nesses anos).
- **CRUZADA_INFORMATIVA** — duas fontes independentes para a mesma quantidade (MapBiomas × PAM); diferença esperada.
- **SEM_GABARITO** — variável sem cobertura no agregado externo nesse ano.
- **ANOMALA** — fora da tolerância sem justificativa documentada. Deve investigar.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIAG = ROOT / "outputs" / "diagnosticos"

df = pd.read_csv(DIAG / "validacao_painel.csv")
resumo = pd.read_csv(DIAG / "validacao_painel_resumo.csv")
print(f"validacao_painel.csv: {df.shape[0]:,} linhas")
print(df['flag'].value_counts())

## 1. Heatmap bloco × ano

% de linhas anômalas por (bloco, ano). Verde = 0% (tudo OK ou esperado); vermelho = ≥1 ANOMALA. Pré-1989 mostra amarelo apenas para blocos com efeito Tocantins, mas como esses são reclassificados como ESPERADA_TOCANTINS, o heatmap fica todo verde.

In [ ]:
pivot = resumo.pivot(index="bloco", columns="ano", values="pct_anomala").fillna(0)
fig, ax = plt.subplots(figsize=(16, max(3, 0.45 * len(pivot))))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=10)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("% de linhas ANOMALA por bloco × ano (0 = limpo)")
plt.colorbar(im, ax=ax, label="% anomala")
plt.tight_layout()
plt.show()

## 2. Séries painel vs gabarito

Linha por variável-chave (bovinos, pastagem, PIB, SICOR total, soja PAM). A reta painel e a reta gabarito devem ficar visualmente sobrepostas pós-1989.

In [ ]:
alvos = [
    ("pec_gabarito", "bovinos_cab",          "Bovinos (cabeças)"),
    ("lulc",         "Pastagem",             "Pastagem (ha)"),
    ("pib",          "pib_real_rs",          "PIB real (R$)"),
    ("sicor",        "total_real_rs",        "SICOR total (R$)"),
    ("agri",         "soja_ha_plantada",     "Soja PAM área plantada (ha)"),
    ("populacao",    "habitantes",           "População"),
]
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
for (bloco, var, titulo), ax in zip(alvos, axes.flat):
    sub = df[(df["bloco"] == bloco) & (df["variavel"] == var)].sort_values("ano")
    ax.plot(sub["ano"], sub["soma_painel"], label="painel (Σ municípios)", lw=2)
    ax.plot(sub["ano"], sub["gabarito"],     label="gabarito UF", lw=1.5, ls="--")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Scatter painel × gabarito (todos os blocos, em escala log)

Pontos sobre a reta y=x indicam batimento perfeito. Cor por flag.

In [ ]:
scat = df[df["gabarito"].notna() & (df["gabarito"] > 0) & (df["soma_painel"] > 0)].copy()
cores = {
    "OK": "#2ca02c",
    "ESPERADA_TOCANTINS": "#ff7f0e",
    "ESPERADA_CENSO": "#9467bd",
    "CRUZADA_INFORMATIVA": "#1f77b4",
    "CRUZADA_DIVERGENTE": "#1f77b4",
    "ANOMALA": "#d62728",
    "SEM_GABARITO": "#7f7f7f",
}
fig, ax = plt.subplots(figsize=(8, 8))
for flag, cor in cores.items():
    s = scat[scat["flag"] == flag]
    if len(s):
        ax.scatter(s["gabarito"], s["soma_painel"], c=cor, s=12, alpha=0.6, label=f"{flag} (n={len(s)})")
vmin = max(1, min(scat["gabarito"].min(), scat["soma_painel"].min()))
vmax = max(scat["gabarito"].max(), scat["soma_painel"].max())
ax.plot([vmin, vmax], [vmin, vmax], "k--", lw=0.8, label="y = x")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("gabarito"); ax.set_ylabel("soma_painel")
ax.set_title("Painel × Gabarito (log–log)")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Distribuição das diferenças relativas

Histograma de `diff_rel` separado por flag. Anomalias (vermelho) ficam fora da janela ±0,5%.

In [ ]:
with_gab = df[df["diff_rel"].notna()].copy()
fig, ax = plt.subplots(figsize=(12, 5))
for flag, cor in cores.items():
    s = with_gab[with_gab["flag"] == flag]["diff_rel"].clip(-0.5, 0.5)
    if len(s):
        ax.hist(s, bins=60, alpha=0.55, label=f"{flag} (n={len(s)})", color=cor)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("diff_rel (clip ±50 %)")
ax.set_ylabel("freq")
ax.set_title("Distribuição de divergência relativa por flag")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Top 20 anomalias (se houver)

Pronto para inspeção manual contra SIDRA web.

In [ ]:
anom = df[df["flag"] == "ANOMALA"].copy()
if len(anom):
    anom["abs_diff_rel"] = anom["diff_rel"].abs()
    top = anom.sort_values("abs_diff_rel", ascending=False).head(20)
    display(top[["bloco", "variavel", "ano", "soma_painel", "gabarito", "diff_rel", "fonte_gabarito"]])
else:
    print("Nenhuma anomalia — todas as divergências são esperadas, OK, ou sem gabarito.")

## 6. Cruzada MapBiomas × PAM — soja

Mesma quantidade (área de soja em GO), dois caminhos: detecção raster (MapBiomas) e declaração ao IBGE (PAM 1612). A razão MapBiomas/PAM tipicamente cai de ~1,4 nos anos 2000 para ~1,1 hoje (melhoria de cobertura PAM).

In [ ]:
cz = df[df["bloco"] == "cruzada"].sort_values("ano")
cz["razao"] = cz["soma_painel"] / cz["gabarito"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(cz["ano"], cz["soma_painel"], label="MapBiomas", lw=2)
axes[0].plot(cz["ano"], cz["gabarito"],    label="PAM 1612",  lw=1.5, ls="--")
axes[0].set_title("Soja em GO — MapBiomas vs PAM (ha)")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(cz["ano"], cz["razao"], lw=2)
axes[1].axhline(1, color="k", lw=0.5)
axes[1].set_title("Razão MapBiomas / PAM")
axes[1].set_ylim(0.7, 1.6); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()